In [316]:
import polars as pl

# Define the path to the data file
file_path = "/media/m2_front/research/data/dexamine/uniswap_v2/usdc_weth/events_usdc_weth.parquet"

# Load data with Polars
try:
    #data = pl.read_parquet(file_path)
    data = pl.read_parquet(file_path).sort(["block_number", "index", "event_index"])
    print("Data loaded successfully. Here is an overview:")
    print(data)
    print("Here are the columns:")
    print(data.columns)
except FileNotFoundError:
    print(f"File not found: {file_path}")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

Data loaded successfully. Here is an overview:
shape: (5_711_114, 30)
┌────────────┬────────────┬───────┬────────────┬───┬────────────┬───────────┬────────────┬─────────┐
│ timestamp  ┆ block_numb ┆ index ┆ event_inde ┆ … ┆ marginal_p ┆ invariant ┆ to_type    ┆ tx_type │
│ ---        ┆ er         ┆ ---   ┆ x          ┆   ┆ rice       ┆ ---       ┆ ---        ┆ ---     │
│ i64        ┆ ---        ┆ i64   ┆ ---        ┆   ┆ ---        ┆ f64       ┆ str        ┆ i64     │
│            ┆ i64        ┆       ┆ i64        ┆   ┆ f64        ┆           ┆            ┆         │
╞════════════╪════════════╪═══════╪════════════╪═══╪════════════╪═══════════╪════════════╪═════════╡
│ 1588712832 ┆ 10008555   ┆ 25    ┆ 6          ┆ … ┆ 206.0      ┆ 0.004854  ┆ dex_router ┆ 0       │
│ 1588712972 ┆ 10008566   ┆ 1     ┆ 4          ┆ … ┆ 205.587587 ┆ 0.004854  ┆ dex_router ┆ 0       │
│ 1588713155 ┆ 10008585   ┆ 2     ┆ 4          ┆ … ┆ 201.486251 ┆ 0.004855  ┆ dex_router ┆ 0       │
│ 1588782295 ┆ 100137

In [317]:
# Forward fill price column 
# "price" is the marginal price of the pool after the swap. "price" is not present in LP events,
# so we need to fill the missing values with the last observed price
print(data.select("marginal_price")[0:5])

#data = data.with_columns(pl.col("price").fill_null(strategy="forward"))
#print(data.select("price")[0:5])

print("Here are desciptive statistics:")
print(data.select("marginal_price").describe())

shape: (5, 1)
┌────────────────┐
│ marginal_price │
│ ---            │
│ f64            │
╞════════════════╡
│ 206.0          │
│ 205.587587     │
│ 201.486251     │
│ 201.078391     │
│ 201.358458     │
└────────────────┘
Here are desciptive statistics:
shape: (9, 2)
┌────────────┬────────────────┐
│ statistic  ┆ marginal_price │
│ ---        ┆ ---            │
│ str        ┆ f64            │
╞════════════╪════════════════╡
│ count      ┆ 5.711114e6     │
│ null_count ┆ 0.0            │
│ mean       ┆ 1963.07711     │
│ std        ┆ 1081.448193    │
│ min        ┆ 26.774068      │
│ 25%        ┆ 1251.297554    │
│ 50%        ┆ 1824.617306    │
│ 75%        ┆ 2751.433261    │
│ max        ┆ 4879.754131    │
└────────────┴────────────────┘


In [318]:
# Transform timestamp to date from unix time
data = data.with_columns(
    pl.from_epoch(pl.col("timestamp"), time_unit="s").alias("clocktime")
)

# Filter for dates after 2021-07-01
data = data.filter(pl.col("clocktime") >= pl.datetime(2020, 7, 1))

print(data.select("clocktime"))

shape: (5_681_900, 1)
┌─────────────────────┐
│ clocktime           │
│ ---                 │
│ datetime[μs]        │
╞═════════════════════╡
│ 2020-07-01 00:00:30 │
│ 2020-07-01 00:02:24 │
│ 2020-07-01 00:04:15 │
│ 2020-07-01 00:05:22 │
│ 2020-07-01 00:05:35 │
│ …                   │
│ 2024-09-19 15:24:35 │
│ 2024-09-19 15:25:11 │
│ 2024-09-19 15:25:35 │
│ 2024-09-19 15:26:47 │
│ 2024-09-19 15:26:59 │
└─────────────────────┘


In [319]:
# Add a new column 'return' that calculates the return based on 'price'
data = data.with_columns(
    ((pl.col("marginal_price") / pl.col("marginal_price").shift(1)) - 1).alias("return")
)

print(data.select("return").describe())


shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ return      │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 5.681899e6  │
│ null_count ┆ 1.0         │
│ mean       ┆ 8.7862e-7   │
│ std        ┆ 0.000964    │
│ min        ┆ -0.345695   │
│ 25%        ┆ -0.000026   │
│ 50%        ┆ -4.6629e-15 │
│ 75%        ┆ 0.000024    │
│ max        ┆ 0.525745    │
└────────────┴─────────────┘


In [320]:
# Filter large returns to examine them more closely
large_returns = data.filter((pl.col("return") > 0.3) | (pl.col("return") < -0.3))
print(large_returns)

# Convert to list and print all hashes
hashes = large_returns.select("tx_hash").to_series().to_list()
print(hashes)
usd_amounts = large_returns.select("amount_0").to_series().to_list()
print(usd_amounts)

# Filter for transactions with large returns around block 14953915
around_returns = data.filter((pl.col("block_number") < 16804970+3) & (pl.col("block_number") > 16804970-3))
print(around_returns)

shape: (4, 32)
┌────────────┬─────────────┬───────┬─────────────┬───┬─────────┬─────────┬─────────────┬───────────┐
│ timestamp  ┆ block_numbe ┆ index ┆ event_index ┆ … ┆ to_type ┆ tx_type ┆ clocktime   ┆ return    │
│ ---        ┆ r           ┆ ---   ┆ ---         ┆   ┆ ---     ┆ ---     ┆ ---         ┆ ---       │
│ i64        ┆ ---         ┆ i64   ┆ i64         ┆   ┆ str     ┆ i64     ┆ datetime[μs ┆ f64       │
│            ┆ i64         ┆       ┆             ┆   ┆         ┆         ┆ ]           ┆           │
╞════════════╪═════════════╪═══════╪═════════════╪═══╪═════════╪═════════╪═════════════╪═══════════╡
│ 1666805315 ┆ 15833742    ┆ 2     ┆ 15          ┆ … ┆ mev     ┆ 2       ┆ 2022-10-26  ┆ 0.40388   │
│            ┆             ┆       ┆             ┆   ┆         ┆         ┆ 17:28:35    ┆           │
│ 1677527195 ┆ 16721702    ┆ 0     ┆ 20          ┆ … ┆ mev     ┆ 2       ┆ 2023-02-27  ┆ 0.406061  │
│            ┆             ┆       ┆             ┆   ┆         ┆         ┆ 1

In [321]:
# Filter for transactions around block 14953915
plot_data = data.filter((pl.col("block_number") < 16804970+10) & (pl.col("block_number") > 16804970-10))

# Plot the returns
import plotly.graph_objects as go
import polars as pl

# Assuming 'data' is your Polars DataFrame with 'timestamp' and 'return' columns
# Convert Polars columns to NumPy arrays
x_values = plot_data.select("clocktime").to_numpy().flatten()
y_values =plot_data.select("marginal_price").to_numpy().flatten()

# Create a figure and add Scattergl trace
fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=x_values,
        y=y_values,
        mode='markers',
        marker=dict(
            line=dict(
                width=1,
                color='DarkSlateGrey'
            )
        )
    )
)

# Display the figure
fig.show()

In [322]:
# Verify that LPs have zero impact on the marginal price (return)
lp_events = data.filter((pl.col("event_type") == "mint") | (pl.col("event_type") == "burn"))
print(lp_events.select("return").describe())

shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ return      │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 60496.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 4.1873e-16  │
│ std        ┆ 7.5779e-14  │
│ min        ┆ -1.2534e-11 │
│ 25%        ┆ -7.7716e-16 │
│ 50%        ┆ 0.0         │
│ 75%        ┆ 2.8866e-15  │
│ max        ┆ 5.7554e-13  │
└────────────┴─────────────┘
